# JUNE 25th-anniversary analysis — Step 2: match the inventory to PubMed

**What this does:** downloads every JUNE record in PubMed and matches each
one to a row of the inventory from Step 1. Matched rows get:
- PubMed and PubMed Central IDs (needed for citation counts in Step 3)
- correct page numbers (many early items had none)
- abstracts and author keywords (for topics in Step 4)
- the publication date (for "citations in the first 5 years")

**What you need to do:** run each cell from top to bottom. It takes about a
minute. Then open `data/needs_review.csv` and fix what you can by adding rows
to `manual_fixes.csv` (see "Hand corrections" below), and run again.

**What goes in:** `data/june_inventory.csv` (from Step 1), `manual_fixes.csv`

**What comes out** (in the `data/` folder):
- `june_matched.csv` — the inventory plus the PubMed information
- `matching_details.csv` — how each row was matched and where its pages and
  DOI came from (for checking; you don't normally need it)
- `needs_review.csv` — **the one file to review.** Every row a human should
  check, with a `reason` column saying why
- `pubmed_june.csv` — every JUNE record in PubMed, for reference

## Running in Google Colab

If you opened this notebook from the "Open in Colab" link, run the next cell
first. It downloads the project's files from GitHub (the CSVs from earlier
steps and `manual_fixes.csv`) so the rest of the notebook can find them.
On your own computer, the cell does nothing.

**Colab forgets everything when the session ends.** To keep a result, download
it from the Files panel (folder icon on the left). Make hand corrections in
`manual_fixes.csv` on GitHub, not in Colab, so they aren't lost.

In [1]:
import os
import subprocess
import sys

if "google.colab" in sys.modules and not os.path.exists("manual_fixes.csv"):
    if not os.path.exists("JUNE_Utilities"):
        subprocess.run(["git", "clone", "https://github.com/ajuavinett/JUNE_Utilities.git"], check=True)
    os.chdir("JUNE_Utilities")
    print("Working in", os.getcwd())

## Settings — the only cell you should normally need to edit

In [2]:
INVENTORY_CSV = "data/june_inventory.csv"
MANUAL_FIXES_CSV = "manual_fixes.csv"

OUTPUT_CSV = "data/june_matched.csv"
REVIEW_CSV = "data/needs_review.csv"
DETAILS_CSV = "data/matching_details.csv"
PUBMED_CSV = "data/pubmed_june.csv"

# How PubMed abbreviates the journal's name
PUBMED_JOURNAL = "J Undergrad Neurosci Educ"

# NCBI asks for an email so they can reach you if something goes wrong (optional)
CONTACT_EMAIL = ""

# How alike two titles must be (0 = nothing in common, 1 = identical)
ACCEPT_SCORE = 0.85   # at or above this, accept the match
REVIEW_SCORE = 0.60   # between this and ACCEPT_SCORE, suggest it for a human to check

## Setup (no need to edit below this line)

In [3]:
import re
import time
import xml.etree.ElementTree as ET
from difflib import SequenceMatcher

import pandas as pd
import requests

EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
NCBI_PARAMS = {"tool": "JUNE-25th-anniversary", "email": CONTACT_EMAIL}
CROSSREF_MEMBER_ID = "38466"   # FUN, JUNE's publisher
PAGE_PREFIX_NAMES = {"A": "article", "E": "editorial", "R": "review",
                     "C": "case study", "T": "technical"}

## Step A: download every JUNE record from PubMed

First we ask PubMed for the list of all JUNE record IDs (PMIDs), then we
download the full records 200 at a time. This is not cached, so each run
picks up newly indexed articles.

In [4]:
def get_all_pmids():
    params = {**NCBI_PARAMS, "db": "pubmed", "term": f'"{PUBMED_JOURNAL}"[jour]',
              "retmax": 10000, "retmode": "json"}
    resp = requests.get(EUTILS + "esearch.fcgi", params=params, timeout=60)
    resp.raise_for_status()
    return resp.json()["esearchresult"]["idlist"]


def text_of(element):
    """All the text inside an XML element, including text in <i>, <sup>, etc."""
    if element is None:
        return ""
    return re.sub(r"\s+", " ", "".join(element.itertext())).strip()


def parse_pubmed_article(article):
    """One <PubmedArticle> → one dict of the fields we need."""
    cit = article.find("MedlineCitation")
    art = cit.find("Article")
    ids = {i.get("IdType"): i.text for i in article.findall("PubmedData/ArticleIdList/ArticleId")}

    # Abstracts can come in labeled parts (BACKGROUND:, METHODS:, …)
    parts = []
    for a in art.findall("Abstract/AbstractText"):
        label = a.get("Label")
        parts.append((label + ": " if label else "") + text_of(a))

    # Publication date: prefer the electronic date, else the issue date
    date = ""
    e = art.find("ArticleDate")
    if e is not None:
        date = "-".join(x.text for x in (e.find("Year"), e.find("Month"), e.find("Day")) if x is not None)
    if not date:
        pd_ = art.find("Journal/JournalIssue/PubDate")
        date = text_of(pd_.find("Year")) or text_of(pd_.find("MedlineDate"))[:4]

    # Authors as "Lom B, Gittis AG" (or a group name, e.g. "FUN Board")
    authors = []
    for a in art.findall("AuthorList/Author"):
        if a.find("LastName") is not None:
            authors.append((text_of(a.find("LastName")) + " " + text_of(a.find("Initials"))).strip())
        elif a.find("CollectiveName") is not None:
            authors.append(text_of(a.find("CollectiveName")))

    return {
        "pmid": text_of(cit.find("PMID")),
        "pubmed_authors": ", ".join(authors),
        "pubmed_first_author": authors[0].rsplit(" ", 1)[0] if authors else "",
        "pmcid": ids.get("pmc", ""),
        "pubmed_doi": ids.get("doi", ""),
        "pubmed_volume": text_of(art.find("Journal/JournalIssue/Volume")),
        "pubmed_issue": text_of(art.find("Journal/JournalIssue/Issue")),
        "pubmed_pages": text_of(art.find("Pagination/MedlinePgn")),
        "pubmed_title": text_of(art.find("ArticleTitle")),
        "pubmed_date": date,
        "pubmed_abstract": " ".join(parts),
        "pubmed_keywords": "; ".join(text_of(k) for k in cit.findall("KeywordList/Keyword")),
        "pubmed_types": "; ".join(text_of(t) for t in art.findall("PublicationTypeList/PublicationType")),
    }


def download_pubmed_records(pmids):
    records = []
    for start in range(0, len(pmids), 200):
        batch = pmids[start:start + 200]
        print(f"  downloading records {start + 1}–{start + len(batch)} of {len(pmids)}")
        resp = requests.post(EUTILS + "efetch.fcgi", timeout=120,
                             data={**NCBI_PARAMS, "db": "pubmed", "id": ",".join(batch), "retmode": "xml"})
        resp.raise_for_status()
        root = ET.fromstring(resp.content)
        records += [parse_pubmed_article(a) for a in root.findall("PubmedArticle")]
        time.sleep(0.5)   # NCBI allows about 3 requests per second
    return pd.DataFrame(records)

## Step B: read PubMed's page numbers

PubMed shortens page ranges: "A60-7" means A60–A67, and "E11-3" means E11–E13.

In [5]:
def expand_pages(pages):
    """'A60-7' → ('A', 60, 67);  'E4' → ('E', 4, 4);  'R1-R5' → ('R', 1, 5)."""
    m = re.match(r"^\s*([A-Za-z]?)(\d+)(?:\s*-\s*[A-Za-z]?(\d+))?", str(pages))
    if not m:
        return None, None, None
    prefix, start, end = (m[1] or "A").upper(), m[2], m[3]
    if end is None:
        end = start
    elif len(end) < len(start):               # "60-7" → the 7 replaces the last digit of 60
        end = start[:len(start) - len(end)] + end
    return prefix, int(start), int(end)

## Step C: match inventory rows to PubMed records

For each inventory row, we look only at PubMed records from the **same
volume**, and try, in order:
1. **DOI** — the same DOI (newer articles only), *and* a similar title
2. **Pages** — same issue and same first page, *and* a similar title
   (the title check catches website linking errors)
3. **Title** — the most similar title in the volume

Titles are compared after removing labels such as "EDITORIAL:" or
"AMAZING PAPERS IN NEUROSCIENCE:", capital letters, and punctuation.
A PubMed record can match only one inventory row.

In [6]:
# Labels in ALL CAPS ("AMAZING PAPERS IN NEUROSCIENCE:") and a few common ones
# in any capitalization ("Commentary:", "Book Review —")
CAPS_LABEL_RE = re.compile(r"^\s*[A-Z][A-Z ’'&/-]{2,}[A-Z]\s*[:—–-]\s*")
WORD_LABEL_RE = re.compile(r"^\s*(commentary|essay|editorial|interview|letter to editor|"
                           r"(book|media|textbook|technology|website) review|of note)\s*[:—–-]\s*", re.I)


def normalize_title(title):
    title = WORD_LABEL_RE.sub("", CAPS_LABEL_RE.sub("", str(title)))
    title = re.sub(r"[^a-z0-9 ]", " ", title.lower())
    return re.sub(r"\s+", " ", title).strip()


def title_similarity(a, b):
    """0 to 1. Also counts one title containing the other (e.g., a title
    plus subtitle on the website but only the title in PubMed)."""
    a, b = normalize_title(a), normalize_title(b)
    if not a or not b:
        return 0.0
    shorter, longer = sorted([a, b], key=len)
    if len(shorter) >= 25 and shorter in longer:
        return 0.95
    return SequenceMatcher(None, a, b).ratio()


def match_inventory(inv, pm):
    pm = pm.copy()
    pm["volume"] = pd.to_numeric(pm["pubmed_volume"], errors="coerce")
    pm["issue"] = pd.to_numeric(pm["pubmed_issue"], errors="coerce")
    pm[["p_prefix", "p_start", "p_end"]] = pm["pubmed_pages"].apply(lambda s: pd.Series(expand_pages(s)))
    doi_to_pmid = {d.lower(): p for d, p in zip(pm["pubmed_doi"], pm["pmid"]) if d}

    results = []
    for r in inv.itertuples():
        same_volume = pm[pm["volume"] == r.volume]
        found = {"pmid": "", "match_method": "", "match_score": None,
                 "suggested_pmid": "", "suggested_title": ""}

        # 1. DOI (+ a looser title check: a few website pages show the wrong DOI)
        if r.doi and r.doi.lower() in doi_to_pmid:
            p = doi_to_pmid[r.doi.lower()]
            s = title_similarity(r.title, pm.loc[pm["pmid"] == p, "pubmed_title"].iloc[0])
            if s >= REVIEW_SCORE:
                found.update(pmid=p, match_method="doi", match_score=round(s, 2))
                results.append(found)
                continue

        scores = [(title_similarity(r.title, t), p) for t, p in zip(same_volume["pubmed_title"], same_volume["pmid"])]
        best_score, best_pmid = max(scores) if scores else (0.0, "")

        # 2. Pages (+ a looser title check)
        if pd.notna(r.page_start):
            same_page = same_volume[(same_volume["issue"] == r.issue) & (same_volume["p_prefix"] == r.page_prefix)
                                    & (same_volume["p_start"] == r.page_start)]
            for t, p in zip(same_page["pubmed_title"], same_page["pmid"]):
                s = title_similarity(r.title, t)
                if s >= REVIEW_SCORE:
                    found.update(pmid=p, match_method="pages", match_score=round(s, 2))
                    break
            if found["pmid"]:
                results.append(found)
                continue

        # 3. Title
        if best_score >= ACCEPT_SCORE:
            found.update(pmid=best_pmid, match_method="title", match_score=round(best_score, 2))
        elif best_score >= REVIEW_SCORE:
            found.update(suggested_pmid=best_pmid, match_score=round(best_score, 2),
                         suggested_title=pm.loc[pm["pmid"] == best_pmid, "pubmed_title"].iloc[0])
        results.append(found)

    out = pd.concat([inv.reset_index(drop=True), pd.DataFrame(results)], axis=1)

    # A PubMed record should match only one row
    claimed = out[out["pmid"] != ""]
    dup = claimed["pmid"].duplicated(keep=False)
    out.loc[claimed.index[dup], "match_method"] += " (PMID matched twice)"
    return out

## Step D: hand corrections (`manual_fixes.csv`)

Some things a program can't decide, so we fix them by hand in
`manual_fixes.csv`. Each row changes one value in one inventory row:

| item_id | title | column | value | note |
|---|---|---|---|---|
| JUNE-14-2-15 | Lights! Camera! Action Projects! | pmid | 12345678 | matched by hand |
| JUNE-03-2-09 | OF NOTE: Brief Reviews | pmid | none | not in PubMed |
| JUNE-13-3-02 | The 2014 FUN Achievement Award | is_fun_workshop | False | award, not a workshop paper |
| JUNE-01-1-01 | Introducing The Journal | reviewed | True | editorial with no byline; fine as is |

- `item_id` comes from the inventory (first column)
- `title` is the start of the item's title. IDs are numbered by position in
  the issue, so if Step 1 ever finds a missed item, later IDs shift. The
  title lets the program warn you when an ID no longer points to that item.
- `column` is any column name in `june_matched.csv`
- `value` is the new value; for `pmid`, write `none` to say "not in PubMed"
- `note` is for people; the program ignores it
- to take a row off the review list without changing anything, set the
  `reviewed` column to `True` (last example above)

Fixes are applied every run, so they're never lost when you re-run.
PMID fixes are applied before PubMed's information is copied in; all other
fixes are applied after, so a hand correction always wins.

In [7]:
def apply_manual_fixes(df, path, pmid_only=False):
    """pmid_only=True applies only the PMID fixes (before PubMed info is copied in);
    pmid_only=False applies the rest (after, so hand fixes have the last word)."""
    try:
        fixes = pd.read_csv(path, dtype=str, keep_default_na=False)
    except FileNotFoundError:
        print(f"No {path} yet — skipping hand corrections.")
        return df
    fixes = fixes[(fixes["column"] == "pmid") == pmid_only]
    for f in fixes.itertuples():
        rows = df.index[df["item_id"] == f.item_id]
        if len(rows) == 0:
            print(f"  ⚠ manual fix for unknown item_id {f.item_id!r} — check the spelling")
            continue
        current_title = df.at[rows[0], "title"]
        if f.title and not current_title.lower().startswith(f.title.strip().lower()):
            print(f"  ⚠ {f.item_id} is now {current_title[:60]!r}, not {f.title!r} — "
                  f"fix skipped; update the item_id in {path}")
            continue
        if f.column not in df.columns:
            print(f"  ⚠ manual fix for unknown column {f.column!r} — check the spelling")
            continue
        value = f.value
        if f.column in ("is_fun_workshop", "reviewed"):
            value = value.strip().lower() in ("true", "yes", "1")
        elif pd.api.types.is_numeric_dtype(df[f.column]):
            value = pd.to_numeric(value)     # e.g., page numbers: "117" → 117
        if f.column == "pmid":
            df.loc[rows, "match_method"] = "hand" if value.lower() != "none" else "hand: not in PubMed"
            value = "" if value.lower() == "none" else value
        df.loc[rows, f.column] = value
    if len(fixes):
        print(f"Applied {len(fixes)} hand corrections from {path}.")
    return df

## Step E: copy PubMed's information into the matched rows

- **Pages:** PubMed's page numbers replace the website's (they're checked
  by the publisher; the website's come from file names)
- **DOI:** PubMed's DOI is kept in its own column; Step F decides which DOI is right
- **Abstract, keywords, authors:** kept from Step 1 if we already had them,
  otherwise taken from PubMed
- **Item type:** filled in from the page prefix if it was missing

In [8]:
def add_pubmed_info(df, pm):
    info = pm.set_index("pmid")
    df["pages_source"] = df["page_start"].notna().map({True: "website", False: ""})
    for i in df.index[df["pmid"] != ""]:
        if df.at[i, "pmid"] not in info.index:
            print(f"  ⚠ {df.at[i, 'item_id']}: PMID {df.at[i, 'pmid']} is not a JUNE record in PubMed")
            continue
        rec = info.loc[df.at[i, "pmid"]]
        prefix, start, end = expand_pages(rec["pubmed_pages"])
        if start is not None:
            df.at[i, "page_prefix"], df.at[i, "page_start"], df.at[i, "page_end"] = prefix, start, end
            df.at[i, "pages_source"] = "PubMed"
        for col in ["pmcid", "pubmed_title", "pubmed_types"]:
            df.at[i, col] = rec[col]
        df.at[i, "publication_date"] = rec["pubmed_date"]
        if not df.at[i, "abstract"]:
            df.at[i, "abstract"] = rec["pubmed_abstract"]
        if not df.at[i, "authors"]:
            df.at[i, "authors"] = rec["pubmed_authors"]
            df.at[i, "first_author"] = df.at[i, "first_author"] or rec["pubmed_first_author"]
        if not df.at[i, "keywords"]:
            df.at[i, "keywords"] = rec["pubmed_keywords"]
        df.at[i, "pubmed_doi"] = rec["pubmed_doi"]
        if not df.at[i, "item_type"] and prefix:
            df.at[i, "item_type"] = PAGE_PREFIX_NAMES.get(prefix, "")
    return df

## Step F: check DOIs against CrossRef

JUNE articles have had DOIs since about 2022. The website and PubMed each
show the **wrong** DOI for a few articles (usually another JUNE article's).
CrossRef is the official DOI registry, so for each row we keep the DOI that
CrossRef lists under a matching title:

1. the website's DOI or PubMed's DOI, whichever CrossRef confirms
2. otherwise, the CrossRef record in the same volume with the most similar title
3. otherwise (older items with no CrossRef record), whatever DOI we had

The `doi_source` column says which one was used.

In [9]:
def get_politely(url, params, tries=4):
    """requests.get, but if the server says "too many requests" (or has a
    hiccup), wait a little longer each time and try again."""
    for attempt in range(1, tries + 1):
        resp = requests.get(url, params=params, timeout=60)
        if resp.status_code not in (429, 500, 502, 503, 504) or attempt == tries:
            resp.raise_for_status()
            return resp
        print(f"  server busy ({resp.status_code}); waiting {10 * attempt} seconds…")
        time.sleep(10 * attempt)


def crossref_records():
    """Every JUNE record in CrossRef: DOI, volume, and title."""
    works, offset = [], 0
    while True:
        resp = get_politely("https://api.crossref.org/works",
                            params={"filter": f"member:{CROSSREF_MEMBER_ID}", "rows": 100, "offset": offset})
        page = resp.json()["message"]["items"]
        works += page
        if len(page) < 100:
            break
        offset += 100
    return pd.DataFrame([{"doi": w["DOI"].lower(), "volume": w.get("volume", ""),
                          "title": re.sub(r"<[^>]+>", "", " ".join(w.get("title", [])))}
                         for w in works])


def check_dois(df, cr):
    cr_title = dict(zip(cr["doi"], cr["title"]))
    df["doi_source"] = ""
    for i in df.index:
        title = df.at[i, "title"]
        candidates = [("website", df.at[i, "doi"]), ("PubMed", df.at[i, "pubmed_doi"])]
        # 1. a DOI we already have, confirmed by CrossRef's title
        confirmed = [(title_similarity(title, cr_title[d.lower()]), src, d)
                     for src, d in candidates if d and d.lower() in cr_title]
        confirmed = [c for c in confirmed if c[0] >= REVIEW_SCORE]
        if confirmed:
            _, src, doi = max(confirmed)
            new_doi, source = doi, f"{src} (confirmed by CrossRef)"
        else:
            # 2. look the title up in CrossRef (same volume)
            same_volume = cr[cr["volume"] == str(df.at[i, "volume"])]
            scores = [(title_similarity(title, t), d) for t, d in zip(same_volume["title"], same_volume["doi"])]
            best = max(scores) if scores else (0, "")
            if best[0] >= ACCEPT_SCORE:
                new_doi, source = best[1], "CrossRef title match"
            else:
                # 3. keep what we had
                src, doi = next(((s, d) for s, d in candidates if d), ("", ""))
                new_doi, source = doi, (f"{src} (not in CrossRef)" if doi else "")
        if df.at[i, "doi"] and new_doi.lower() != df.at[i, "doi"].lower():
            print(f"  {df.at[i, 'item_id']}: website DOI {df.at[i, 'doi']} → {new_doi} ({source})")
        df.at[i, "doi"], df.at[i, "doi_source"] = new_doi, source
    return df

## Step G: the review list (`needs_review.csv`)

One file collects everything a human should check. The `reason` column says why:
- **no PubMed match** — expected for most media reviews and for the newest
  issue (PubMed takes a few months); otherwise worth a look. If there's a
  suggested match, `suggested_title` shows it.
- **PMID matched to two rows** — two inventory rows claim the same PubMed record
- **same page as another item** — two items in the same issue start on the same page
- **notes from Step 1** — e.g., no authors found, or a section that doesn't
  match the page prefix
- **in PubMed but not in the inventory** — an item Step 1 missed (these rows
  have no `item_id`; add them to `EXTRA_ITEMS` in Step 1)

Rows marked `reviewed = True` in `manual_fixes.csv` are left off.

In [10]:
def build_review_list(df, pm):
    reasons = [[] for _ in range(len(df))]
    same_page = df.duplicated(subset=["volume", "issue", "page_prefix", "page_start"], keep=False) \
        & df["page_start"].notna()
    for n, (i, r) in enumerate(df.iterrows()):
        if r["pmid"] == "" and r["match_method"] != "hand: not in PubMed":
            reasons[n].append("no PubMed match")
        if "twice" in r["match_method"]:
            reasons[n].append("PMID matched to two rows")
        if same_page[i]:
            reasons[n].append("same page as another item")
        # Notes from Step 1, minus any a hand fix has since resolved
        for note in filter(None, r["flags"].split("; ")):
            if note == "no authors found" and r["authors"]:
                continue
            reasons[n].append(note)
    df = df.assign(reason=["; ".join(x) for x in reasons])
    review = df[(df["reason"] != "") & ~df["reviewed"].astype(bool)]

    cols = ["item_id", "reason", "volume", "issue", "section", "item_type", "title", "authors",
            "page_prefix", "page_start", "pmid", "match_method", "match_score",
            "suggested_pmid", "suggested_title", "pdf_url", "article_page_url"]
    review = review[cols]

    # PubMed records that no inventory row matched
    missing = pm[~pm["pmid"].isin(df["pmid"])]
    missing_rows = pd.DataFrame({
        "item_id": "", "reason": "in PubMed but not in the inventory",
        "volume": missing["pubmed_volume"], "issue": missing["pubmed_issue"],
        "title": missing["pubmed_title"], "page_prefix": missing["pubmed_pages"],
        "pmid": missing["pmid"],
        "article_page_url": "https://pubmed.ncbi.nlm.nih.gov/" + missing["pmid"] + "/",
    })
    return pd.concat([review, missing_rows], ignore_index=True).astype("string").fillna("")

## Step H: links

- `doi_url` — the DOI as a link. JUNE DOIs start with issue 20(2) (2022);
  earlier items don't have one.
- `pmc_url` — the free full text on PubMed Central
- `url` — one link per item, the first of: DOI link, article page, PDF,
  PubMed Central. DOI links never break; website addresses can change.

In [11]:
def add_links(df):
    df["doi_url"] = ["https://doi.org/" + d if d else "" for d in df["doi"]]
    df["pmc_url"] = [f"https://pmc.ncbi.nlm.nih.gov/articles/{p}/" if p else "" for p in df["pmcid"]]
    df["url"] = [next((u for u in links if u), "")
                 for links in zip(df["doi_url"], df["article_page_url"], df["pdf_url"], df["pmc_url"])]
    return df

## Step I: what goes in each output file

`june_matched.csv` keeps only the columns later steps and readers need.
The bookkeeping from matching (scores, PubMed's own title and DOI, where
the pages came from, …) goes to `matching_details.csv` instead, so nothing
is lost. Hand fixes can still change any column.

In [12]:
MATCHED_COLUMNS = [
    # what the item is
    "item_id", "volume", "issue", "year_label", "section", "item_type", "series",
    "is_fun_workshop", "workshop_name",
    # content
    "title", "authors", "first_author", "reviewed_work_author", "abstract", "keywords",
    # where it is
    "page_prefix", "page_start", "page_end", "publication_date",
    "doi", "pmid", "pmcid", "url", "doi_url", "pmc_url", "pdf_url", "article_page_url", "issue_url",
    # other
    "n_supplements", "listed_on",
]
DETAILS_COLUMNS = [
    "item_id", "title", "match_method", "match_score", "suggested_pmid", "suggested_title",
    "pubmed_title", "pubmed_doi", "pubmed_types", "pages_source", "doi_source", "flags", "reviewed",
]


def save_matched(df):
    df[MATCHED_COLUMNS].to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    df[DETAILS_COLUMNS].to_csv(DETAILS_CSV, index=False, encoding="utf-8-sig")

## Run everything

In [13]:
def match_to_pubmed():
    inv = pd.read_csv(INVENTORY_CSV, keep_default_na=False, encoding="utf-8-sig")
    for col in ["page_start", "page_end"]:
        inv[col] = pd.to_numeric(inv[col], errors="coerce").astype("Int64")
    print(f"Read {len(inv)} inventory rows.")

    pmids = get_all_pmids()
    print(f"PubMed has {len(pmids)} JUNE records.")
    pm = download_pubmed_records(pmids)
    pm.to_csv(PUBMED_CSV, index=False, encoding="utf-8-sig")

    df = match_inventory(inv, pm)
    for col in ["pmcid", "pubmed_title", "publication_date", "pubmed_types", "pubmed_doi"]:
        df[col] = ""
    df["reviewed"] = False
    df = apply_manual_fixes(df, MANUAL_FIXES_CSV, pmid_only=True)
    df = add_pubmed_info(df, pm)
    df = check_dois(df, crossref_records())
    df = apply_manual_fixes(df, MANUAL_FIXES_CSV, pmid_only=False)
    df = add_links(df)
    df["page_start"] = pd.to_numeric(df["page_start"], errors="coerce").astype("Int64")
    df["page_end"] = pd.to_numeric(df["page_end"], errors="coerce").astype("Int64")

    save_matched(df)

    review = build_review_list(df, pm)
    review.to_csv(REVIEW_CSV, index=False, encoding="utf-8-sig")

    print(f"\nMatched {(df['pmid'] != '').sum()} of {len(df)} inventory rows → {OUTPUT_CSV}")
    print(f"{len(review)} rows to check by hand → {REVIEW_CSV}")
    print(review["reason"].str.split("; ").explode().value_counts().to_string())
    return df, pm


matched, pubmed = match_to_pubmed()
matched.head(10)

Read 716 inventory rows.


PubMed has 656 JUNE records.
  downloading records 1–200 of 656


  downloading records 201–400 of 656


  downloading records 401–600 of 656


  downloading records 601–656 of 656


  server busy (429); waiting 10 seconds…


  JUNE-20-2-04: website DOI 10.59390/LKPG6213 → 10.59390/TPQZ7702 (PubMed (confirmed by CrossRef))
  JUNE-20-2-11: website DOI 10.5939/KYSI6629 → 10.59390/kysi6629 (CrossRef title match)
  JUNE-20-2-15: website DOI 10.59390/BKPW4729 → 10.59390/LVJD2796 (PubMed (confirmed by CrossRef))
  JUNE-21-2-08: website DOI 10.59390/JVIC5712 → 10.59390/AXNC2524 (PubMed (confirmed by CrossRef))
Applied 4 hand corrections from manual_fixes.csv.

Matched 656 of 716 inventory rows → data/june_matched.csv
62 rows to check by hand → data/needs_review.csv
reason
no PubMed match                         60
no authors found                         1
no PDF link                              1
section 'Articles' but page prefix R     1


,item_id,year_label,volume,issue,section,series,item_type,is_fun_workshop,workshop_name,title,...,pubmed_title,publication_date,pubmed_types,pubmed_doi,reviewed,pages_source,doi_source,doi_url,pmc_url,url
0,JUNE-01-1-01,2002,1,1,Editorials,,editorial,False,,Introducing The Journal of Undergraduate Neuro...,...,Introducing The Journal of Undergraduate Neuro...,2002-10-15,Journal Article,,False,PubMed,,,https://pmc.ncbi.nlm.nih.gov/articles/PMC3592582/,https://www.funjournal.org/wp-content/uploads/...
1,JUNE-01-1-02,2002,1,1,Articles,,article,False,,The Emergence of Non-Match-to-Sample Behavior ...,...,The Emergence of Non-Match-to-Sample Behavior ...,2002-10-15,Journal Article,,False,PubMed,,,https://pmc.ncbi.nlm.nih.gov/articles/PMC3592587/,https://www.funjournal.org/wp-content/uploads/...
2,JUNE-01-1-03,2002,1,1,Articles,,article,False,,And The Winner Is: Inviting Hollywood into the...,...,And the winner is: inviting hollywood into the...,2002-10-15,Journal Article,,False,PubMed,,,https://pmc.ncbi.nlm.nih.gov/articles/PMC3592583/,https://www.funjournal.org/wp-content/uploads/...
3,JUNE-01-1-04,2002,1,1,Articles,,article,False,,Exploring Neurogenesis in Crustaceans,...,Exploring neurogenesis in crustaceans.,2002-10-15,Journal Article,,False,PubMed,,,https://pmc.ncbi.nlm.nih.gov/articles/PMC3592584/,https://www.funjournal.org/wp-content/uploads/...
4,JUNE-01-1-05,2002,1,1,Articles,,article,False,,Action Potentials in a Giant Algal Cell: A Com...,...,Action potentials in a giant algal cell: a com...,2002-10-15,Journal Article,,False,PubMed,,,https://pmc.ncbi.nlm.nih.gov/articles/PMC3592585/,https://www.funjournal.org/wp-content/uploads/...
5,JUNE-01-1-06,2002,1,1,Media Reviews,Book review,review,False,,Book Review: Mood Genes,...,,,,,False,website,,,,https://www.funjournal.org/wp-content/uploads/...
6,JUNE-01-1-07,2002,1,1,Media Reviews,Book review,review,False,,Textbook Review: Biological Psychology,...,,,,,False,website,,,,https://www.funjournal.org/wp-content/uploads/...
7,JUNE-01-1-08,2002,1,1,Media Reviews,Media review,review,False,,Media Review: Crawdad: A CD-ROM Lab Manual for...,...,,,,,False,website,,,,https://www.funjournal.org/wp-content/uploads/...
8,JUNE-01-1-09,2002,1,1,Media Reviews,Media review,review,False,,Media Review: The Secret Life of the Brain,...,,,,,False,website,,,,https://www.funjournal.org/wp-content/uploads/...
9,JUNE-01-2-01,2003,1,2,Editorials,,editorial,False,,Introductory Neuroscience – Courses in an Evol...,...,Introductory neuroscience-courses in an evolvi...,2003-06-15,Journal Article,,False,PubMed,,,https://pmc.ncbi.nlm.nih.gov/articles/PMC3670786/,https://www.funjournal.org/wp-content/uploads/...


## Check the results

Things to look for:
- volumes where few rows matched (a sign Step 1 read those pages oddly)
- `data/needs_review.csv`, especially any "in PubMed but not in the inventory" rows
- media reviews usually aren't in PubMed, so unmatched reviews are expected

In [14]:
# How each row was matched
matched["match_method"].replace("", "(no match)").value_counts()

match_method
pages         401
doi           135
title         120
(no match)     60
Name: count, dtype: int64

In [15]:
# Share of rows matched, by item type
(matched.assign(matched=matched["pmid"] != "")
 .groupby(matched["item_type"].replace("", "(unknown)"))["matched"].agg(["sum", "count"])
 .rename(columns={"sum": "matched", "count": "rows"}))

,matched,rows
item_type,,
article,483,492
case study,13,13
editorial,113,116
review,44,92
technical,3,3


In [16]:
# Rows matched and rows with an abstract, by volume
(matched.assign(matched=matched["pmid"] != "", has_abstract=matched["abstract"] != "")
 .groupby("volume")[["matched", "has_abstract"]].sum()
 .assign(rows=matched.groupby("volume").size()))

,matched,has_abstract,rows
volume,,,
1,12,9,18
2,13,10,17
3,15,11,16
4,17,12,19
5,22,12,24
6,16,12,18
7,16,12,17
8,26,22,28
9,20,15,20
